# 04 - Backbone Fine-Tuning vs Freezing

Compare the same backbone with and without backbone training. The frozen run uses cached backbone embeddings with `ArcFaceHeadModel`, while the unfrozen run fine-tunes the full `ArcFaceModel` end-to-end.

In [1]:
import sys
from pathlib import Path

import pandas as pd
import torch
from dotenv import load_dotenv

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import src.data as data
import src.feature_cache as feature_cache
import src.inference as inference
import src.wandb_utils as wandb_utils
from src.models import ArcFaceHeadModel, ArcFaceModel, build_backbone
from src.training import build_optimizer, fit, fit_embeddings
from src.utils import get_device, set_seed

load_dotenv(project_root / ".env")

RANDOM_SEED = 42
set_seed(RANDOM_SEED)

device = get_device()
device

device(type='cuda')

## Base Config

In [ ]:
config = {
    # Paths
    "data_dir": Path("../data"),
    "checkpoint_dir": Path("../checkpoints/e4_backbone_finetune"),
    "submission_dir": Path("../submissions/e4_backbone_finetune"),
    "embeddings_cache_dir": Path("../checkpoints/embedding_cache"),

    # Model
    "backbone_name": "eva02_large_patch14_448.mim_m38m_ft_in22k_in1k",
    "input_size": 448,
    "embedding_dim": 256,
    "hidden_dim": 512,

    # ArcFace
    "arcface_margin": 0.5,
    "arcface_scale": 64.0,
    "dropout": 0.3,

    # Training
    "batch_size": 32,
    "learning_rate": 1e-4,
    "head_learning_rate": 1e-4,
    "backbone_learning_rate": 1e-5,
    "weight_decay": 1e-4,
    "num_epochs": 100,
    "patience": 8,
    "val_split": 0.2,
    "num_workers": 2,
    "augment_train": False,
    "train_last_n_layers": 0,

    # Reproducibility
    "seed": RANDOM_SEED,
}

config["rerank"] = {
    "enabled": True,
    "k1": 20,
    "k2": 6,
    "lambda_value": 0.3,
}

config["checkpoint_dir"].mkdir(parents=True, exist_ok=True)
config["submission_dir"].mkdir(parents=True, exist_ok=True)
config["embeddings_cache_dir"].mkdir(parents=True, exist_ok=True)
config

## Load Data

In [ ]:
def load_data(config, backbone):
    train_df = data.load_train_df(config["data_dir"])
    train_df, label_encoder = data.encode_labels(train_df)
    num_classes = len(label_encoder.classes_)

    train_data, val_data = data.train_val_split(
        train_df,
        val_split=config["val_split"],
        seed=config["seed"],
        stratify_col="ground_truth",
    )

    backbone_train_loader, backbone_val_loader = data.create_backbone_dataloaders(
        backbone,
        train_data,
        val_data,
        img_dir=config["data_dir"] / "train" / "train",
        input_size=config["input_size"],
        batch_size=config["batch_size"],
        num_workers=config["num_workers"],
    )

    train_loader, val_loader = data.create_dataloaders(
        train_data,
        val_data,
        img_dir=config["data_dir"] / "train" / "train",
        input_size=config["input_size"],
        batch_size=config["batch_size"],
        num_workers=config["num_workers"],
        mean=config["mean"],
        std=config["std"],
        weighted_sampling=False,
    )

    pairs_df = data.load_test_pairs_df(config["data_dir"])
    unique_images = sorted(set(pairs_df["query_image"].unique()) | set(pairs_df["gallery_image"].unique()))
    test_df = pd.DataFrame({"filename": unique_images})

    test_loader = data.create_test_loader(
        backbone,
        test_df,
        img_dir=config["data_dir"] / "test" / "test",
        input_size=config["input_size"],
        batch_size=config["batch_size"],
        num_workers=config["num_workers"],
    )

    print(
        f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Backbone train batches: {len(backbone_train_loader)} | Backbone val batches: {len(backbone_val_loader)} | Test batches: {len(test_loader)}"
    )

    return {
        "label_encoder": label_encoder,
        "num_classes": num_classes,
        "backbone_train_loader": backbone_train_loader,
        "backbone_val_loader": backbone_val_loader,
        "train_loader": train_loader,
        "val_loader": val_loader,
        "pairs_df": pairs_df,
        "test_loader": test_loader,
    }


## Run Experiment

We train 5 models:
- Fine-Tune last 2 Blocks of Backbone
- Fine-Tune last 4 Blocks of Backbone
- Fine-Tune last 8 Blocks of Backbone
- Fine-Tune complete Backbone
- Freeze Backbone

In [ ]:
experiments = [
    {
        "name": "Backbone_train_last_2",
        "freeze_backbone": False,
        "train_last_n_layers": 2,
        "backbone_learning_rate": 1e-5,
    },
    {
        "name": "Backbone_train_last_4",
        "freeze_backbone": False,
        "train_last_n_layers": 4,
        "backbone_learning_rate": 1e-5,
    },
    {
        "name": "Backbone_train_last_8",
        "freeze_backbone": False,
        "train_last_n_layers": 8,
        "backbone_learning_rate": 1e-5,
    },
    {
        "name": "Backbone_finetune_all",
        "freeze_backbone": False,
        "train_last_n_layers": 0,
        "backbone_learning_rate": 1e-5,
    },
    {
        "name": "Backbone_frozen",
        "freeze_backbone": True,
        "train_last_n_layers": 0,
        "backbone_learning_rate": 0.0,
    },
]


def run_experiment(experiment, run_idx, total_runs):
    print("=" * 80)
    print(f"Run {run_idx}/{total_runs}: {experiment['name']}")

    config["backbone_name"] = experiment["backbone_name"]
    config["input_size"] = experiment["input_size"]
    config["freeze_backbone"] = experiment["freeze_backbone"]
    config["train_last_n_layers"] = experiment.get("train_last_n_layers", experiment.get("freeze_last_n_layers", 0))
    config["backbone_learning_rate"] = experiment["backbone_learning_rate"]

    train_df = data.load_train_df(config["data_dir"])
    train_df, label_encoder = data.encode_labels(train_df)
    num_classes = len(label_encoder.classes_)

    if config["freeze_backbone"]:
        backbone = build_backbone(config["backbone_name"], pretrained=True).to(device)
        backbone.eval()
        backbone_out_dim = getattr(backbone, "num_features", None)
        if backbone_out_dim is None:
            raise ValueError("Backbone output dimension not found; pass backbone_out_dim")
        print(f"Backbone fully frozen for cached head training: {config['backbone_name']}")
        data_bundle = load_data(config, backbone)
    else:
        model = ArcFaceModel(
            backbone_name=config["backbone_name"],
            num_classes=num_classes,
            embedding_dim=config["embedding_dim"],
            hidden_dim=config["hidden_dim"],
            margin=config["arcface_margin"],
            scale=config["arcface_scale"],
            dropout=config["dropout"],
            pretrained=True,
            freeze_backbone=config["freeze_backbone"],
            train_last_n_layers=config["train_last_n_layers"],
        ).to(device)
        print(model)
        if hasattr(model.backbone, "blocks"):
            total_backbone_blocks = len(model.backbone.blocks)
            trainable_blocks = list(range(max(0, total_backbone_blocks - config["train_last_n_layers"]), total_backbone_blocks)) if config["train_last_n_layers"] > 0 else list(range(total_backbone_blocks))
            frozen_blocks = [idx for idx in range(total_backbone_blocks) if idx not in trainable_blocks]
            print(f"Backbone blocks total: {total_backbone_blocks}")
            print(f"Trainable backbone blocks: {trainable_blocks}")
            print(f"Frozen backbone blocks: {frozen_blocks}")
            trainable_extra = [name for name in ("norm", "fc_norm") if hasattr(model.backbone, name) and config["train_last_n_layers"] > 0]
            if trainable_extra:
                print(f"Also trainable: {[f'backbone.{name}' for name in trainable_extra]}")
        backbone = model.backbone
        data_bundle = load_data(config, backbone)

    label_encoder = data_bundle["label_encoder"]
    num_classes = data_bundle["num_classes"]
    pairs_df = data_bundle["pairs_df"]
    test_loader = data_bundle["test_loader"]

    if config["freeze_backbone"]:
        cache_key = f"{config['backbone_name'].replace(':', '_').replace('/', '_')}_{config['input_size']}"
        train_cache = config["embeddings_cache_dir"] / f"train_{cache_key}.npz"
        val_cache = config["embeddings_cache_dir"] / f"val_{cache_key}.npz"

        train_embeddings, train_labels = feature_cache.get_or_create_embeddings(
            train_cache,
            backbone,
            data_bundle["backbone_train_loader"],
            device,
        )
        val_embeddings, val_labels = feature_cache.get_or_create_embeddings(
            val_cache,
            backbone,
            data_bundle["backbone_val_loader"],
            device,
        )

        train_loader, val_loader = data.create_embedding_dataloaders(
            train_embeddings,
            train_labels,
            val_embeddings,
            val_labels,
            batch_size=config["batch_size"],
        )

        model = ArcFaceHeadModel(
            input_dim=backbone_out_dim,
            num_classes=num_classes,
            embedding_dim=config["embedding_dim"],
            hidden_dim=config["hidden_dim"],
            margin=config["arcface_margin"],
            scale=config["arcface_scale"],
            dropout=config["dropout"],
        ).to(device)
    else:
        train_loader = data_bundle["train_loader"]
        val_loader = data_bundle["val_loader"]

    if model.arcface.num_classes != num_classes:
        raise ValueError(f"Model num_classes={model.arcface.num_classes} but dataset has {num_classes}")

    total_params = sum(p.numel() for p in model.parameters()) + (sum(p.numel() for p in backbone.parameters()) if config["freeze_backbone"] else 0)
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    backbone_params = sum(p.numel() for p in backbone.parameters())

    wandb = wandb_utils.init_wandb(
        config,
        run_name=experiment["name"],
        param_count=trainable_params,
        param_count_backbone=backbone_params,
    )
    wandb.run.summary["total_param_count"] = total_params
    wandb.run.summary["trainable_param_count"] = trainable_params

    criterion = torch.nn.CrossEntropyLoss()
    optimizer = build_optimizer(model, config)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=2,
    )

    checkpoint_name = f"{experiment['name']}.pth"
    checkpoint_path = config["checkpoint_dir"] / checkpoint_name
    if config["freeze_backbone"]:
        results = fit_embeddings(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            config=config,
            device=device,
            criterion=criterion,
            optimizer=optimizer,
            scheduler=scheduler,
            label_encoder=label_encoder,
            wandb_run=wandb,
            checkpoint_name=checkpoint_name,
        )
    else:
        results = fit(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            config=config,
            device=device,
            criterion=criterion,
            optimizer=optimizer,
            scheduler=scheduler,
            label_encoder=label_encoder,
            wandb_run=wandb,
            checkpoint_name=checkpoint_name,
        )

    wandb.run.summary["best_val_mAP"] = results["best_map"]
    wandb.run.summary["best_val_mAP_rerank"] = results.get("best_map_rerank")
    wandb.run.summary["best_val_loss"] = results["best_val_loss"]
    wandb.run.summary["best_epoch"] = results["best_epoch"]
    wandb.run.summary["total_epochs"] = results["epochs_ran"]

    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    plain_submission_path = config["submission_dir"] / f"submission_{experiment['name']}.csv"
    rerank_submission_path = config["submission_dir"] / f"submission_{experiment['name']}_rerank.csv"

    if config["freeze_backbone"]:
        inference.create_submission_backbone(
            backbone,
            model,
            device,
            pairs_df,
            test_loader,
            output_path=plain_submission_path,
            use_rerank=False,
        )

        inference.create_submission_backbone(
            backbone,
            model,
            device,
            pairs_df,
            test_loader,
            output_path=rerank_submission_path,
            use_rerank=config["rerank"]["enabled"],
            k1=config["rerank"]["k1"],
            k2=config["rerank"]["k2"],
            lambda_value=config["rerank"]["lambda_value"],
        )
    else:
        inference.create_submission_model(
            model,
            device,
            pairs_df,
            test_loader,
            output_path=plain_submission_path,
            use_rerank=False,
        )

        inference.create_submission_model(
            model,
            device,
            pairs_df,
            test_loader,
            output_path=rerank_submission_path,
            use_rerank=config["rerank"]["enabled"],
            k1=config["rerank"]["k1"],
            k2=config["rerank"]["k2"],
            lambda_value=config["rerank"]["lambda_value"],
        )

    wandb_utils.log_checkpoint_artifact(
        wandb,
        checkpoint_path,
        artifact_name=experiment["name"],
        description="checkpoint",
    )

    wandb_utils.log_submission_artifact(
        wandb,
        plain_submission_path,
        artifact_name=f"{experiment['name']}_plain",
        description="Competition submission file",
    )

    wandb_utils.log_submission_artifact(
        wandb,
        rerank_submission_path,
        artifact_name=f"{experiment['name']}_rerank",
        description="Competition submission file with reranking",
    )

    wandb.finish()
    print(f"Finished run {run_idx}")


for run_idx, experiment in enumerate(experiments, start=1):
    run_experiment(experiment, run_idx, len(experiments))


View the runs on W&B: [W&B Run Group](https://wandb.ai/juggling-jaguars/jaguar-reid-jugglingjaguars/groups/Experiment-4-BackboneFinetuning)

![](../images/e4_wandb_dashboard.png)